<a href="https://colab.research.google.com/github/ineverydomain/RF_PROJECT_COLAB/blob/main/RF_Signal_Classification_GPU_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Oct  8 15:29:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_DRIVE = "/content/drive/MyDrive/RF_SIGNAL_PROJECT"

os.makedirs(PROJECT_DRIVE, exist_ok=True)

print(PROJECT_DRIVE)

/content/drive/MyDrive/RF_SIGNAL_PROJECT


In [6]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [17]:
import zipfile
import os

zip_path = "/content/RF SIGNAL ML.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

print(os.listdir("/content/RF SIGNAL ML"))

['plots', 'generate_requirements.py', 'test_gen.py', 'models', '__pycache__', '.kilo', 'rf_signal_classification_resumable.py', 'requirements.txt', 'cache', 'rf_signal_classification.py']


In [18]:
!cp -r /content/RF_SIGNAL_PROJECT /content/drive/MyDrive/

In [19]:
!rm -rf /content/RF_SIGNAL_PROJECT
!cp -r /content/drive/MyDrive/RF_SIGNAL_PROJECT /content/

In [21]:
!find /content/RF_SIGNAL_PROJECT -maxdepth 2 -type f | sort

/content/RF_SIGNAL_PROJECT/cache/snrbase_v2_spc15_s777.npz
/content/RF_SIGNAL_PROJECT/cache/test_v2_spc60_s200.npz
/content/RF_SIGNAL_PROJECT/cache/train_v2_spc150_s100.npz
/content/RF_SIGNAL_PROJECT/generate_requirements.py
/content/RF_SIGNAL_PROJECT/.kilo/.gitignore
/content/RF_SIGNAL_PROJECT/models/rf_classifier.pth
/content/RF_SIGNAL_PROJECT/__pycache__/rf_signal_classification.cpython-313.pyc
/content/RF_SIGNAL_PROJECT/requirements.txt
/content/RF_SIGNAL_PROJECT/rf_signal_classification.py
/content/RF_SIGNAL_PROJECT/rf_signal_classification_resumable.py
/content/RF_SIGNAL_PROJECT/test_gen.py


In [23]:
!pip install -q torchsig scipy scikit-learn matplotlib pandas h5py python-docx

In [24]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.11.0+cu130
True
13.0


In [26]:
from torchsig.datasets.datasets import TorchSigIterableDataset
from torchsig.utils.defaults import TorchSigDefaults

print("TorchSig imported successfully")
print(TorchSigDefaults().default_dataset_metadata.keys())

TorchSig imported successfully
dict_keys(['num_iq_samples_dataset', 'num_signals_min', 'num_signals_max', 'fft_size', 'fft_stride', 'sample_rate', 'noise_power_db', 'snr_db_min', 'snr_db_max', 'cochannel_overlap_probability', 'signal_duration_in_samples_min', 'signal_duration_in_samples_max', 'bandwidth_min', 'bandwidth_max', 'signal_center_freq_min', 'signal_center_freq_max', 'frequency_min', 'frequency_max'])


In [27]:
%cd /content/RF_SIGNAL_PROJECT


/content/RF_SIGNAL_PROJECT


In [28]:
!python rf_signal_classification_resumable.py --help

usage: rf_signal_classification_resumable.py [-h] [--skip-plots]
                                             [--force-plots] [--force-data]
                                             [--epochs EPOCHS]
                                             [--batch-size BATCH_SIZE]
                                             [--train-spc TRAIN_SPC]
                                             [--test-spc TEST_SPC] [--lr LR]
                                             [--skip-train] [--resume]
                                             [--new-data] [--skip-docx]
                                             [--num-workers NUM_WORKERS]

RF AMC pipeline + academic report

options:
  -h, --help            show this help message and exit
  --skip-plots
  --force-plots
  --force-data
  --epochs EPOCHS
  --batch-size BATCH_SIZE
  --train-spc TRAIN_SPC
  --test-spc TEST_SPC
  --lr LR
  --skip-train
  --resume              Resume training from the previous full checkpoint
  --new-data            Gen

In [31]:
!python rf_signal_classification_resumable.py --epochs 1 --skip-plots

  RF SIGNAL CLASSIFICATION — HYBRID AMC PIPELINE
  Classes: 46  |  PyTorch: 2.11.0+cu130

>>> STEP 1–2: Skipping plots (0 PNGs; use --force-plots)
    [1/8] fm
    [2/8] ofdm-64
    [3/8] bpsk
    [4/8] 2fsk
    [5/8] qpsk
    [6/8] am-dsb
    [7/8] 64qam
    [8/8] 16qam

>>> STEP 3: Multi-SNR datasets
    Loading cache: train_v2_spc150_s100.npz
    Loading cache: test_v2_spc60_s200.npz
    Loading cache: snrbase_v2_spc15_s777.npz
    GPU 0: Tesla T4  free=15.5GB / 15.6GB
    Using: Tesla T4

>>> STEP 4: Training HybridAMCNet
    Parameters: 1,539,470
    Train: 6900  Test: 2760  epochs_this_run=1  bs=64
    Epoch 001  train=0.128  val=0.205  loss=3.055/2.557
    Saved resumable checkpoint → /content/RF_SIGNAL_PROJECT/models/rf_classifier_checkpoint.pth
    Cumulative epochs: 1  |  Best val=0.205 @ epoch 1  (19.8s)

>>> STEP 5: Evaluation
    Test accuracy: 0.2054  macro-F1: 0.1327
    Family accuracy: 0.4848
    SNR sweep...
      SNR  -4 dB → acc=0.051
      SNR  +0 dB → acc=0.116
  

In [33]:
!python rf_signal_classification_resumable.py --epochs 2 --skip-plots

  RF SIGNAL CLASSIFICATION — HYBRID AMC PIPELINE
  Classes: 46  |  PyTorch: 2.11.0+cu130

>>> STEP 1–2: Skipping plots (1 PNGs; use --force-plots)
    [1/8] ofdm-64
    [2/8] am-dsb
    [3/8] qpsk
    [4/8] fm
    [5/8] 16qam
    [6/8] bpsk
    [7/8] 64qam
    [8/8] 2fsk

>>> STEP 3: Multi-SNR datasets
    Loading cache: train_v2_spc150_s100.npz
    Loading cache: test_v2_spc60_s200.npz
    Loading cache: snrbase_v2_spc15_s777.npz
    GPU 0: Tesla T4  free=15.5GB / 15.6GB
    Using: Tesla T4

>>> STEP 4: Training HybridAMCNet
    Parameters: 1,539,470
    Train: 6900  Test: 2760  epochs_this_run=2  bs=64
    Epoch 001  train=0.128  val=0.205  loss=3.055/2.557
    Epoch 002  train=0.255  val=0.271  loss=2.337/2.408
    Saved resumable checkpoint → /content/RF_SIGNAL_PROJECT/models/rf_classifier_checkpoint.pth
    Cumulative epochs: 2  |  Best val=0.271 @ epoch 2  (34.2s)

>>> STEP 5: Evaluation
    Test accuracy: 0.2707  macro-F1: 0.2310
    Family accuracy: 0.5366
    SNR sweep...
    

In [35]:
!python rf_signal_classification_resumable.py

  RF SIGNAL CLASSIFICATION — HYBRID AMC PIPELINE
  Classes: 46  |  PyTorch: 2.11.0+cu130

>>> STEP 1: Clean reference signals
    [1/46] ofdm-128
    [2/46] ook
    [3/46] 32ask
    [4/46] 8ask
    [5/46] qpsk
    [6/46] ofdm-72
    [7/46] ofdm-64
    [8/46] bpsk
    [9/46] 16psk
    [10/46] 64ask
    [11/46] 8msk
    [12/46] 64qam
    [13/46] 16msk
    [14/46] 4gfsk
    [15/46] 4fsk
    [16/46] 16ask
    [17/46] am-dsb
    [18/46] 16gfsk
    [19/46] 2gmsk
    [20/46] am-usb
    [21/46] am-dsb-sc
    [22/46] 16fsk
    [23/46] ofdm-1024
    [24/46] 2fsk
    [25/46] fm
    [26/46] tone
    [27/46] 16qam
    [28/46] 8fsk
    [29/46] 4ask
    [30/46] 8gfsk
    [31/46] 16gmsk
    [32/46] 256qam
    [33/46] 32psk
    [34/46] ofdm-300
    [35/46] ofdm-512
    [36/46] 128qam_cross
    [37/46] am-lsb
    [38/46] 2gfsk
    [39/46] 2msk
    [40/46] ofdm-256
    [41/46] 4msk
    [42/46] 4gmsk
    [43/46] 8psk
    [44/46] 32qam
    [45/46] 64psk
    [46/46] 8gmsk
    Collected 46/46 in 2.5s

>>> ST

In [36]:
!python rf_signal_classification_resumable_free_colab.py \
    --epochs 2 \
    --skip-plots \
    --drive-backup-dir /content/drive/MyDrive/RF_SIGNAL_PROJECT/models

  RF SIGNAL CLASSIFICATION — HYBRID AMC PIPELINE
  Classes: 46  |  PyTorch: 2.11.0+cu130
  Checkpoints: saved after every completed epoch

>>> STEP 1–2: Skipping plots (1012 PNGs; use --force-plots)
    [1/8] am-dsb
    [2/8] qpsk
    [3/8] 16qam
    [4/8] bpsk
    [5/8] 2fsk
    [6/8] 64qam
    [7/8] ofdm-64
    [8/8] fm

>>> STEP 3: Multi-SNR datasets
    Drive backup enabled → /content/drive/MyDrive/RF_SIGNAL_PROJECT/models
    Loading cache: train_v2_spc150_s100.npz
    Loading cache: test_v2_spc60_s200.npz
    Loading cache: snrbase_v2_spc15_s777.npz
    GPU 0: Tesla T4  free=15.5GB / 15.6GB
    Using: Tesla T4

>>> STEP 4: Training HybridAMCNet
    Parameters: 1,539,470
    Train: 6900  Test: 2760  epochs_this_run=2  bs=64
    Epoch 001  train=0.128  val=0.205  loss=3.055/2.557
    [CHECKPOINT] Epoch 001 saved → /content/RF_SIGNAL_PROJECT/models/rf_classifier_checkpoint.pth
    [DRIVE BACKUP] Epoch 001 → /content/drive/MyDrive/RF_SIGNAL_PROJECT/models
    Epoch 002  train=0.255  

In [37]:
!python rf_signal_classification_resumable_free_colab.py \
    --resume \
    --epochs 100 \
    --skip-plots \
    --drive-backup-dir /content/drive/MyDrive/RF_SIGNAL_PROJECT/models

  RF SIGNAL CLASSIFICATION — HYBRID AMC PIPELINE
  Classes: 46  |  PyTorch: 2.11.0+cu130
  Checkpoints: saved after every completed epoch

>>> STEP 1–2: Skipping plots (1012 PNGs; use --force-plots)
    [1/8] ofdm-64
    [2/8] qpsk
    [3/8] 16qam
    [4/8] 64qam
    [5/8] 2fsk
    [6/8] fm
    [7/8] am-dsb
    [8/8] bpsk

>>> STEP 3: Multi-SNR datasets
    Drive backup enabled → /content/drive/MyDrive/RF_SIGNAL_PROJECT/models
    Reusing training-data round: 0 (seed=100)
    Loading cache: train_v2_spc150_s100.npz
    Loading cache: test_v2_spc60_s200.npz
    Loading cache: snrbase_v2_spc15_s777.npz
    GPU 0: Tesla T4  free=15.5GB / 15.6GB
    Using: Tesla T4

>>> STEP 4: Training HybridAMCNet
    Parameters: 1,539,470
    Train: 6900  Test: 2760  epochs_this_run=100  bs=64
    Resuming checkpoint: /content/RF_SIGNAL_PROJECT/models/rf_classifier_checkpoint.pth
    Continuing: epochs 3 → 102
    Epoch 003  train=0.347  val=0.355  loss=2.085/1.959
    [CHECKPOINT] Epoch 003 saved → /co

In [39]:
!cp -r /content/RF_SIGNAL_PROJECT/rf_signal_classification_resumable_free_colab.py /content/drive/MyDrive/RF_SIGNAL_PROJECT/